# Hands-on Workshop on Transformers — Version 2
## Fine-tune a Pretrained Transformer with Hugging Face

**Dataset:** AG News text classification  
**Model:** DistilBERT  
**Works on:** Google Colab, Kaggle Notebook, and local Jupyter Notebook

Version 2 shows how Transformers are used in real workflows.


```text
Tokenization → Embeddings + Position → Encoder Self-Attention → Linear → Softmax
```

We use a pretrained Transformer encoder and fine-tune it for news classification.


# 0. plan

| Section | Goal |
|---|---|
| Setup | Colab/Kaggle/local compatibility |
| Dataset | Load AG News |
| Tokenizer | Connect to PPT tokenization |
| Pretrained model | Explain encoder + classification head |
| Fine-tuning | Train DistilBERT |
| Evaluation | Metrics and confusion matrix |
| Inference | Custom predictions |
| Attention visualization + tasks | Explore model internals |


# 1. Platform setup

For Colab: `Runtime → Change runtime type → GPU`  
For Kaggle: `Notebook Settings → Accelerator → GPU` and enable Internet  
For local Jupyter:

```bash
pip install torch transformers datasets accelerate pandas scikit-learn matplotlib tqdm
```


In [8]:
import sys, subprocess, importlib.util

packages = {
    "torch": "torch",
    "transformers": "transformers",
    "datasets": "datasets",
    "accelerate": "accelerate",
    "pandas": "pandas",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
    "tqdm": "tqdm",
}

missing = [pip_name for import_name, pip_name in packages.items()
           if importlib.util.find_spec(import_name) is None]

if missing:
    print("Installing:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing)
else:
    print("All required packages are already installed.")




All required packages are already installed.


# 2. Imports and device


In [14]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import warnings
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from datasets import Dataset
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from tqdm.auto import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoConfig,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
)

SEED = 42

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

warnings.filterwarnings("ignore")
try:
    from datasets import disable_progress_bars
    disable_progress_bars()
except Exception:
    pass

try:
    from datasets.utils import logging as datasets_logging
    datasets_logging.set_verbosity_error()
except Exception:
    pass

try:
    from huggingface_hub import logging as hf_logging
    hf_logging.set_verbosity_error()
except Exception:
    pass

Device: cuda
GPU: Tesla T4


# 3. Run mode


In [3]:
RUN_MODE = "fast"  # "debug", "fast", "high_gpu"

CONFIGS = {
    "debug": dict(train_samples=1000, val_samples=300, test_samples=1000,
                  max_length=96, batch_size=16, epochs=1, lr=2e-5),
    "fast": dict(train_samples=12000, val_samples=3000, test_samples=4000,
                         max_length=128, batch_size=32, epochs=2, lr=2e-5),
    "high_gpu": dict(train_samples=50000, val_samples=10000, test_samples=7600,
                     max_length=160, batch_size=64, epochs=3, lr=2e-5),
}
cfg = CONFIGS[RUN_MODE]
cfg


{'train_samples': 12000,
 'val_samples': 3000,
 'test_samples': 4000,
 'max_length': 128,
 'batch_size': 32,
 'epochs': 2,
 'lr': 2e-05}

# 4. Load AG News dataset

Classes:

```text
0 = World
1 = Sports
2 = Business
3 = Sci/Tech
```


In [10]:
def load_ag_news_dataframe():
    try:
        from datasets import load_dataset

        # Public dataset, no token required
        ds = load_dataset("ag_news")

        train_df = pd.DataFrame(ds["train"])[["text", "label"]]
        test_df = pd.DataFrame(ds["test"])[["text", "label"]]

        return train_df, test_df, "Hugging Face ag_news"

    except Exception as e:
        print("Could not load AG News:", repr(e))
        print("Using tiny fallback dataset only to keep notebook executable.")

        examples = {
            0: [
                "The government announced a new international policy.",
                "Foreign leaders met for peace talks."
            ],
            1: [
                "The team won the football match.",
                "The player scored in the final game."
            ],
            2: [
                "The stock market closed higher.",
                "The company reported record profit."
            ],
            3: [
                "Scientists released a new AI model.",
                "A new satellite was launched into orbit."
            ],
        }

        rows = []
        for _ in range(500):
            for label, texts in examples.items():
                for text in texts:
                    rows.append({"text": text, "label": label})

        df = pd.DataFrame(rows).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
        split = int(0.8 * len(df))

        return (
            df.iloc[:split].reset_index(drop=True),
            df.iloc[split:].reset_index(drop=True),
            "tiny fallback"
        )


train_df, test_df, DATA_SOURCE = load_ag_news_dataframe()

train_df = train_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
test_df = test_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

val_df = train_df.iloc[:cfg["val_samples"]].reset_index(drop=True)
train_df = train_df.iloc[
    cfg["val_samples"] : cfg["val_samples"] + cfg["train_samples"]
].reset_index(drop=True)

test_df = test_df.iloc[:cfg["test_samples"]].reset_index(drop=True)

label_names = ["World", "Sports", "Business", "Sci/Tech"]

print("Data source:", DATA_SOURCE)
print("Train:", train_df.shape, "Val:", val_df.shape, "Test:", test_df.shape)

train_df.head()

Data source: Hugging Face ag_news
Train: (12000, 2) Val: (3000, 2) Test: (4000, 2)


,text,label
0,"Dozens killed in Iraq violence as Egypt, Brita...",0
1,"Russians Rally Against Terror, Bury Dead MOSCO...",0
2,"Cavaliers 92, Nuggets 73 The LeBron-Carmelo ma...",1
3,Bush says N. Korea #39;s neighbors uniting aga...,0
4,Eli Manning replaces Warner at quarterback Eli...,1


# 5. Inspect examples


In [11]:
for i in range(5):
    print("=" * 100)
    print("Label:", label_names[int(train_df.loc[i, "label"])])
    print(train_df.loc[i, "text"])


Label: World
Dozens killed in Iraq violence as Egypt, Britain seek to free &lt;b&gt;...&lt;/b&gt; FALLUJAH, Iraq : US airstrikes on rebel-held Fallujah left 15 dead while an insurgent attack in another troubled Sunni Arab town killed 10 more, as Britain and Egypt stepped up efforts to secure the release of hostages in Iraq.
Label: World
Russians Rally Against Terror, Bury Dead MOSCOW - Tens of thousands of people answered a government call and rallied outside the Kremlin on Tuesday in a show of solidarity against terrorism, nearly a week after militants seized a school in southern Russia in a standoff that claimed more than 350 lives, many of them children.    Mourners in the grief-stricken city of Beslan lowered caskets into the damp earth in a third day of burials from the siege, which officials have blamed on Chechens and other Islamic militants...
Label: Sports
Cavaliers 92, Nuggets 73 The LeBron-Carmelo matchup wasn #39;t much of a contest, and neither was the one between the Cava

# 6. Load pretrained tokenizer

PPT relation: **Tokenization**

Modern Transformers use subword tokenization. This helps with rare and unknown words.


In [15]:
MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

sample_text = "Researchers developed a new Transformer model for medical image analysis."
encoded = tokenizer(sample_text)

print("Model:", MODEL_NAME)
print("Text:", sample_text)
print("Token IDs:", encoded["input_ids"])
print("Tokens:", tokenizer.convert_ids_to_tokens(encoded["input_ids"]))


Model: distilbert-base-uncased
Text: Researchers developed a new Transformer model for medical image analysis.
Token IDs: [101, 6950, 2764, 1037, 2047, 10938, 2121, 2944, 2005, 2966, 3746, 4106, 1012, 102]
Tokens: ['[CLS]', 'researchers', 'developed', 'a', 'new', 'transform', '##er', 'model', 'for', 'medical', 'image', 'analysis', '.', '[SEP]']


# 7. Tokenize the dataset


```text
Text → Tokenizer → Input IDs + Attention Mask
```


In [16]:
def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, max_length=cfg["max_length"])

train_hf = Dataset.from_pandas(train_df)
val_hf = Dataset.from_pandas(val_df)
test_hf = Dataset.from_pandas(test_df)

train_tok = train_hf.map(tokenize_batch, batched=True)
val_tok = val_hf.map(tokenize_batch, batched=True)
test_tok = test_hf.map(tokenize_batch, batched=True)

for ds_name in ["train_tok", "val_tok", "test_tok"]:
    ds = globals()[ds_name]
    keep = ["input_ids", "attention_mask", "label"]
    globals()[ds_name] = ds.remove_columns([c for c in ds.column_names if c not in keep])

train_tok = train_tok.rename_column("label", "labels")
val_tok = val_tok.rename_column("label", "labels")
test_tok = test_tok.rename_column("label", "labels")

train_tok.set_format("torch")
val_tok.set_format("torch")
test_tok.set_format("torch")

collator = DataCollatorWithPadding(tokenizer=tokenizer)

train_loader = DataLoader(train_tok, batch_size=cfg["batch_size"], shuffle=True, collate_fn=collator)
val_loader = DataLoader(val_tok, batch_size=cfg["batch_size"], shuffle=False, collate_fn=collator)
test_loader = DataLoader(test_tok, batch_size=cfg["batch_size"], shuffle=False, collate_fn=collator)

batch = next(iter(train_loader))
print({k: v.shape for k, v in batch.items()})


{'labels': torch.Size([32]), 'input_ids': torch.Size([32, 77]), 'attention_mask': torch.Size([32, 77])}


# 8. Load pretrained Transformer model


```text
Input IDs → Embeddings + Position → Transformer Encoder → Linear → Softmax
```

DistilBERT is an encoder-only Transformer model.


In [17]:
config = AutoConfig.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    id2label={i: label_names[i] for i in range(4)},
    label2id={label_names[i]: i for i in range(4)},
)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, config=config).to(DEVICE)

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Trainable parameters:", count_params(model))


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Trainable parameters: 66956548


# 9. Forward pass


In [18]:
model.eval()
with torch.no_grad():
    mini_batch = {k: v.to(DEVICE) for k, v in batch.items()}
    outputs = model(**mini_batch)
    logits = outputs.logits

print("Logits shape:", logits.shape)
print("Predicted class before training:", label_names[int(logits[0].argmax())])


Logits shape: torch.Size([32, 4])
Predicted class before training: World


# 10. Training and evaluation functions


In [19]:
def train_one_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss = 0.0
    preds_all, labels_all = [], []
    for batch in tqdm(loader, desc="Training", leave=False):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        optimizer.zero_grad(set_to_none=True)
        outputs = model(**batch)
        loss = outputs.loss
        logits = outputs.logits
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item() * batch["input_ids"].size(0)
        preds_all.extend(logits.argmax(dim=1).detach().cpu().tolist())
        labels_all.extend(batch["labels"].detach().cpu().tolist())
    return total_loss / len(loader.dataset), accuracy_score(labels_all, preds_all)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    preds_all, labels_all = [], []
    for batch in tqdm(loader, desc="Evaluating", leave=False):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        logits = outputs.logits
        total_loss += loss.item() * batch["input_ids"].size(0)
        preds_all.extend(logits.argmax(dim=1).cpu().tolist())
        labels_all.extend(batch["labels"].cpu().tolist())
    return total_loss / len(loader.dataset), accuracy_score(labels_all, preds_all), labels_all, preds_all


# 11. Fine-tune DistilBERT


In [20]:
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=0.01)

total_steps = len(train_loader) * cfg["epochs"]
warmup_steps = int(0.1 * total_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

history = []
best_val_acc = 0.0
best_path = "best_distilbert_agnews.pt"

for epoch in range(1, cfg["epochs"] + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, scheduler)
    val_loss, val_acc, _, _ = evaluate(model, val_loader)

    history.append(dict(epoch=epoch, train_loss=train_loss, train_acc=train_acc,
                        val_loss=val_loss, val_acc=val_acc))

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), best_path)

    print(f"Epoch {epoch}/{cfg['epochs']} | "
          f"Train loss {train_loss:.4f}, acc {train_acc:.4f} | "
          f"Val loss {val_loss:.4f}, acc {val_acc:.4f}")

history_df = pd.DataFrame(history)
history_df


Training:   0%|          | 0/375 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/94 [00:00<?, ?it/s]

Epoch 1/2 | Train loss 0.4948, acc 0.8375 | Val loss 0.2658, acc 0.9143


Training:   0%|          | 0/375 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/94 [00:00<?, ?it/s]

Epoch 2/2 | Train loss 0.2173, acc 0.9324 | Val loss 0.2362, acc 0.9257


,epoch,train_loss,train_acc,val_loss,val_acc
0,1,0.494792,0.837500,0.265839,0.914333
1,2,0.217260,0.932417,0.236239,0.925667


# 12. Plot training


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history_df["epoch"], history_df["train_acc"], marker="o", label="train_acc")
plt.plot(history_df["epoch"], history_df["val_acc"], marker="o", label="val_acc")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("DistilBERT Fine-tuning: Accuracy")
plt.legend()
plt.grid(True)
plt.show()


# 13. Test evaluation


In [ ]:
model.load_state_dict(torch.load(best_path, map_location=DEVICE))
test_loss, test_acc, y_true, y_pred = evaluate(model, test_loader)

print("Test loss:", round(test_loss, 4))
print("Test accuracy:", round(test_acc, 4))
print(classification_report(y_true, y_pred, target_names=label_names))
print("Confusion matrix:")
print(confusion_matrix(y_true, y_pred))


# 14. predictions


In [ ]:
@torch.no_grad()
def predict_news(text):
    model.eval()
    enc = tokenizer(text, truncation=True, max_length=cfg["max_length"], return_tensors="pt")
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    outputs = model(**enc)
    probs = torch.softmax(outputs.logits, dim=1).squeeze(0).cpu().numpy()
    pred = int(np.argmax(probs))
    return {"text": text, "prediction": label_names[pred],
            "probabilities": {label_names[i]: float(probs[i]) for i in range(4)}}

examples = [
    "The national football team won the tournament after a dramatic final.",
    "The prime minister met foreign leaders to discuss climate policy.",
    "The company reported higher revenue after strong sales growth.",
    "A new quantum computing breakthrough was announced by researchers.",
]
for ex in examples:
    print("=" * 100)
    print(predict_news(ex))


# 15. Attention visualization


In [ ]:
attn_config = AutoConfig.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    output_attentions=True,
    id2label={i: label_names[i] for i in range(4)},
    label2id={label_names[i]: i for i in range(4)},
)

attn_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, config=attn_config).to(DEVICE)
attn_model.load_state_dict(torch.load(best_path, map_location=DEVICE))
attn_model.eval()

@torch.no_grad()
def visualize_transformer_attention(text, layer_idx=-1, head_idx=0):
    enc = tokenizer(text, truncation=True, max_length=cfg["max_length"], return_tensors="pt")
    tokens = tokenizer.convert_ids_to_tokens(enc["input_ids"][0])
    enc = {k: v.to(DEVICE) for k, v in enc.items()}

    outputs = attn_model(**enc, output_attentions=True)
    attn = outputs.attentions[layer_idx][0, head_idx].detach().cpu().numpy()

    L = len(tokens)
    plt.figure(figsize=(9, 7))
    plt.imshow(attn[:L, :L], aspect="auto")
    plt.colorbar(label="attention weight")
    plt.xticks(range(L), tokens, rotation=45, ha="right")
    plt.yticks(range(L), tokens)
    plt.xlabel("Key tokens")
    plt.ylabel("Query tokens")
    plt.title(f"DistilBERT Attention | layer {layer_idx}, head {head_idx}")
    plt.tight_layout()
    plt.show()

    probs = torch.softmax(outputs.logits, dim=1).squeeze(0).cpu().numpy()
    print("Prediction:", label_names[int(np.argmax(probs))])

visualize_transformer_attention(
    "Scientists announced a new artificial intelligence system for medical diagnosis.",
    layer_idx=-1,
    head_idx=0,
)


# 16. freeze the encoder

This is useful when GPU is weak or the dataset is small.  
Only the classification head is trained.


In [ ]:
RUN_FREEZE_EXPERIMENT = False

if RUN_FREEZE_EXPERIMENT:
    frozen_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=4).to(DEVICE)
    for name, param in frozen_model.named_parameters():
        if "classifier" not in name and "pre_classifier" not in name:
            param.requires_grad = False
    print("Trainable parameters after freezing:", count_params(frozen_model))
else:
    print("Skipped freeze experiment.")


# 17. hands-on tasks

## Task 1: Inspect subword tokenization

Run:

```python
tokenizer.convert_ids_to_tokens(tokenizer("your sentence")["input_ids"])
```

Try technical words, names, misspellings, and long words.

Question: how does subword tokenization handle rare words?

---

## Task 2: Compare attention heads

Run:

```python
visualize_transformer_attention("your sentence", layer_idx=-1, head_idx=0)
visualize_transformer_attention("your sentence", layer_idx=-1, head_idx=1)
visualize_transformer_attention("your sentence", layer_idx=-1, head_idx=2)
```

Question: do different heads focus on different tokens?

---

## Task 3: Try another pretrained model

Change:

```python
MODEL_NAME = "distilbert-base-uncased"
```

to one of:

```text
bert-base-uncased
prajjwal1/bert-tiny
google/electra-small-discriminator
```

Question: which model is faster and which gives better accuracy?
